# Distributed Training with Checkpointing

This example demonstrates how to add **checkpoint saving and resumption** to distributed PyTorch training using [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html) and [Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/overview/).

## Why Checkpointing Matters

Long-running distributed training jobs are vulnerable to interruptions:
- **Spot/preemptible instances**: Cloud providers can reclaim these at any time (AWS Spot, Azure Spot VMs, GCP Preemptible)
- **Hardware failures**: GPUs, nodes, or network components can fail during multi-day training runs
- **Resource preemption**: Cluster schedulers may preempt lower-priority jobs

Without checkpointing, an interruption means starting training from scratch -- wasting hours or days of compute. This notebook shows how to periodically save training state and resume from the latest checkpoint.

## What You Will Learn

- Save model weights, optimizer state, and training progress at regular intervals
- Resume training seamlessly from a checkpoint after interruption
- Use the rank-0 save pattern with `dist.barrier()` for safe distributed checkpointing
- Run checkpointed training locally and scale it to multiple nodes with Kubeflow TrainJob

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Install the PyTorch Dependencies

You also need to install PyTorch and Torchvision to be able to run the example locally:

In [ ]:
!pip install torch==2.9.1
!pip install torchvision==0.22.1

## Define the Training Function with Checkpointing

The training function trains a ResNet-18 model on CIFAR-10 with periodic checkpoint saving. Key checkpointing features:

1. **Save checkpoint every N steps** (rank 0 only) -- includes model weights, optimizer state, epoch, and step
2. **Resume from existing checkpoint** -- automatically detects and loads the latest checkpoint on startup
3. **Distributed barrier after save** -- ensures all ranks wait for the checkpoint to be written before continuing

The checkpoint contains:
- `model_state_dict`: Model weights (unwrapped from DDP via `model.module.state_dict()`)
- `optimizer_state_dict`: Optimizer state (learning rates, momentum buffers)
- `epoch` and `step`: Training progress markers for resumption
- `loss`: Last recorded loss value for monitoring

In [ ]:
def train_with_checkpointing():
    import os

    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, transforms
    from torchvision.models import resnet18

    # ── Configuration ──────────────────────────────────────────────────
    num_epochs = 3
    batch_size = 128
    learning_rate = 0.1
    momentum = 0.9
    save_every = 100  # Save a checkpoint every N training steps
    checkpoint_dir = "/tmp/checkpoints"
    # For production with Kubernetes PersistentVolumeClaims (PVCs), use a
    # shared volume path instead, e.g.:
    #   checkpoint_dir = "/mnt/shared-checkpoint-volume/checkpoints"
    # This ensures checkpoints survive pod restarts and can be read by all nodes.
    checkpoint_path = os.path.join(checkpoint_dir, "checkpoint_latest.pt")

    # ── Distributed Setup ──────────────────────────────────────────────
    # Use NCCL if a GPU is available, otherwise use Gloo as communication backend.
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    # Setup PyTorch distributed.
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # Create the device handle for this worker.
    device = torch.device(f"{device}:{local_rank}")

    # ── Model ──────────────────────────────────────────────────────────
    # Use ResNet-18 adapted for CIFAR-10 (32x32 images, 10 classes).
    # Standard ResNet-18 expects 224x224 input. We adapt the first layer:
    #   - 3x3 conv with stride 1 and padding 1 (instead of 7x7 stride 2)
    #   - Remove the initial max pooling layer
    model = resnet18(num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()

    model = nn.parallel.DistributedDataParallel(model.to(device))
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)

    # ── Resume from Checkpoint ─────────────────────────────────────────
    start_epoch = 0
    start_step = 0
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
        model.module.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"]
        start_step = ckpt["step"] + 1
        print(
            f"[Rank {dist.get_rank()}] Resumed from checkpoint: "
            f"epoch {start_epoch}, step {start_step}, loss {ckpt['loss']:.4f}"
        )
    else:
        print(f"[Rank {dist.get_rank()}] No checkpoint found. Starting from scratch.")

    # ── Dataset ────────────────────────────────────────────────────────
    transform = transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ]
    )

    # Download CIFAR-10 dataset only on local_rank=0 process.
    if local_rank == 0:
        dataset = datasets.CIFAR10(
            "./data", train=True, download=True, transform=transform
        )
    dist.barrier()
    dataset = datasets.CIFAR10(
        "./data", train=True, download=False, transform=transform
    )

    # Shard the dataset across workers.
    sampler = DistributedSampler(dataset)
    train_loader = DataLoader(
        dataset, batch_size=batch_size, sampler=sampler, num_workers=2
    )

    # ── Create checkpoint directory (rank 0 only) ─────────────────────
    if dist.get_rank() == 0:
        os.makedirs(checkpoint_dir, exist_ok=True)
    dist.barrier()

    # ── Training Loop ──────────────────────────────────────────────────
    for epoch in range(start_epoch, num_epochs):
        model.train()
        # Ensure each epoch has different shuffling across ranks.
        sampler.set_epoch(epoch)

        for step, (inputs, labels) in enumerate(train_loader):
            # If resuming mid-epoch, skip steps we already completed.
            if epoch == start_epoch and step < start_step:
                continue

            # Copy data to the device.
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass.
            outputs = model(inputs)
            loss = F.cross_entropy(outputs, labels)

            # Backward pass.
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # ── Checkpoint Save ────────────────────────────────────
            if dist.get_rank() == 0 and (step + 1) % save_every == 0:
                torch.save(
                    {
                        "epoch": epoch,
                        "step": step,
                        "model_state_dict": model.module.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "loss": loss.item(),
                    },
                    checkpoint_path,
                )
                print(
                    f"Checkpoint saved at epoch {epoch}, step {step}, "
                    f"loss {loss.item():.4f}"
                )
            # All ranks wait for rank 0 to finish saving.
            dist.barrier()

            # Log training progress.
            if step % 50 == 0 and dist.get_rank() == 0:
                print(
                    "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        epoch,
                        step * len(inputs),
                        len(train_loader.dataset),
                        100.0 * step / len(train_loader),
                        loss.item(),
                    )
                )

        # Reset start_step after the first (possibly partial) epoch.
        start_step = 0

    # ── Save Final Checkpoint ──────────────────────────────────────────
    if dist.get_rank() == 0:
        torch.save(
            {
                "epoch": num_epochs,
                "step": 0,
                "model_state_dict": model.module.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": loss.item(),
            },
            checkpoint_path,
        )
        print(f"Final checkpoint saved. Training loss: {loss.item():.4f}")

    # Wait for all ranks to complete.
    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")

    # Clean up PyTorch distributed.
    dist.destroy_process_group()

## Run the Training Locally

We can submit the training function to the local Trainer client to run it in an isolated subprocess. This is useful for testing the checkpointing logic before scaling to multiple nodes.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# List available runtimes
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

# Submit training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_with_checkpointing,
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

# Stream logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

## Scale PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes.

`TrainerClient()` verifies that you have required access to the Kubernetes cluster.

Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.

### Persistent Checkpoints with PVC

For production use, mount a Kubernetes `PersistentVolumeClaim` (PVC) so checkpoints persist across pod restarts. Change `checkpoint_dir` in the training function to point to the PVC mount path (e.g., `/mnt/shared-checkpoint-volume/checkpoints`).

With a `ReadWriteMany` PVC, all nodes can read the checkpoint for resume. With `ReadWriteOnce`, only the rank-0 pod needs write access; other pods load the checkpoint via DDP's broadcast mechanism after rank 0 loads it.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

Additionally, it might show available accelerator type and number of available resources.

In [ ]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on 2 PyTorch nodes with checkpointing enabled.

Each node trains on its shard of CIFAR-10 and rank 0 saves periodic checkpoints. If the job is interrupted and restarted, it automatically resumes from the latest checkpoint.

In [ ]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_with_checkpointing,
        # Set how many PyTorch nodes you want to use for distributed training.
        num_nodes=2,
        # Set the resources for each PyTorch node.
        resources_per_node={
            "cpu": 4,
            "memory": "8Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes.
            # "nvidia.com/gpu": 1,
        },
    ),
    runtime=torch_runtime,
)

## Check the TrainJob Steps

You can check the components of TrainJob that were created.

Since the TrainJob performs distributed training across 2 nodes, it generates 2 steps: `trainer-node-0` and `trainer-node-1`.

You can get the individual status for each of these steps.

In [ ]:
# Wait for the running status.
client.wait_for_job_status(name=job_name, status={"Running"})

In [ ]:
for c in client.get_job(name=job_name).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

## Watch the TrainJob Logs

We can use the `get_job_logs()` API to get the TrainJob logs.

Look for "Checkpoint saved at epoch" messages in the output to confirm checkpointing is working. If the job resumes from a checkpoint, you will see "Resumed from checkpoint" at the start.

In [ ]:
for logline in client.get_job_logs(job_name, follow=True):
    print(logline)

## Delete the TrainJob

When TrainJob is finished, you can delete the resource.

In [ ]:
# client.delete_job(job_name)